# CS4241 - Introduction to Artificial Intelligence
## Part B: Custom Retrieval System

**Name:** Maureen Amago  
**Index Number:** 10022200180

### Approach
We build a complete retrieval pipeline from scratch using only **Python standard library + NumPy**:
- **Custom TF-IDF Embedding Pipeline** — converts text to vectors (implemented from scratch)
- **Custom Vector Store** — stores all chunk vectors in memory
- **Cosine Similarity** — measures how relevant a chunk is to a query
- **Query Expansion** — fixes the failure case where informal words miss formal document terms

---
## Step 1: Import Libraries
We only use libraries that come built-in with Python and Anaconda — no extra installs needed.

In [17]:
# Name: Maureen Amago | Index: 10022200180
import re
import math
import numpy as np
import pandas as pd
from collections import Counter
from pypdf import PdfReader

print('All libraries imported successfully!')

All libraries imported successfully!


---
## Step 2: Load and Prepare Data (from Part A)

In [18]:
# Name: Maureen Amago | Index: 10022200180

# Extract text from the first 30 pages of the PDF
reader = PdfReader('2025-Budget-Statement-and-Economic-Policy_v4.pdf')
raw_text = ''
for i in range(min(30, len(reader.pages))):
    page_text = reader.pages[i].extract_text()
    if page_text:
        raw_text += page_text + ' '

# Clean the text
clean_text = re.sub(r'\s+', ' ', raw_text)
clean_text = re.sub(r'[^\x00-\x7F]+', ' ', clean_text).strip()

# Create 500-character chunks with 50-character overlap (Strategy 1 from Part A)
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(clean_text)
print(f'Total chunks: {len(chunks)}')
print(f'\nExample chunk:\n{chunks[10]}')

Total chunks: 137

Example chunk:
bility Management ................................ ................ 61 2025 Key Initiatives and Outlook for the Ghana Stock Exchange ................................ . 63 Medium-Term Vision and Policy Objectives ................................ ................................ .. 64 SECTION 5: KEY POLICY INITIATIVES AND RESOURCE ALLOCATION .................... 69 SECTION 6: SECTORAL PERFORMANCE AND OUTLOOK ................................ ......... 74 Introduction ...............................


---
## Step 3: Embedding Pipeline (Custom TF-IDF from Scratch)

**TF-IDF** stands for *Term Frequency - Inverse Document Frequency*.
- **TF (Term Frequency)**: How often does a word appear in this specific chunk?
- **IDF (Inverse Document Frequency)**: How rare is this word across ALL chunks? Rare words are more meaningful.
- **TF-IDF = TF × IDF**: A high score means the word is frequent in this chunk AND rare globally — very distinctive!

We implement this completely from scratch using only Python and NumPy.

In [19]:
# Name: Maureen Amago | Index: 10022200180

# Common English stop words to ignore
STOP_WORDS = set([
    'a', 'an', 'the', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'is', 'are', 'was', 'were', 'be', 'been',
    'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could',
    'should', 'may', 'might', 'it', 'its', 'this', 'that', 'these', 'those',
    'as', 'not', 'no', 'so', 'if', 'than', 'then', 'also', 'into', 'more'
])

def tokenize(text):
    """Convert a text string into a list of clean tokens (words)."""
    text = text.lower()
    tokens = re.findall(r'\b[a-z]{2,}\b', text)  # only words with 2+ letters
    return [t for t in tokens if t not in STOP_WORDS]


class TFIDFEmbedder:
    """
    Custom TF-IDF Embedding Pipeline.
    Converts text documents into numerical vectors.
    """
    def __init__(self, max_vocab=3000):
        self.max_vocab = max_vocab  # limit vocabulary size to keep vectors manageable
        self.vocab = {}             # word -> index mapping
        self.idf = {}              # word -> IDF score

    def fit(self, documents):
        """Learn the vocabulary and IDF scores from the documents."""
        print('Building vocabulary and computing IDF scores...')
        N = len(documents)
        doc_freq = Counter()  # how many documents contain each word
        all_tokens = []

        for doc in documents:
            tokens = set(tokenize(doc))  # unique tokens per document
            doc_freq.update(tokens)
            all_tokens.extend(tokens)

        # Build vocabulary: top max_vocab most common words across all documents
        vocab_words = [word for word, _ in Counter(all_tokens).most_common(self.max_vocab)]
        self.vocab = {word: idx for idx, word in enumerate(vocab_words)}

        # Compute IDF score for each vocabulary word
        for word in self.vocab:
            df = doc_freq.get(word, 0)
            self.idf[word] = math.log((N + 1) / (df + 1)) + 1  # smoothed IDF

        print(f'Vocabulary size: {len(self.vocab)} words')
        return self

    def transform(self, documents):
        """Convert a list of documents into a TF-IDF matrix (numpy array)."""
        V = len(self.vocab)
        matrix = np.zeros((len(documents), V))

        for i, doc in enumerate(documents):
            tokens = tokenize(doc)
            token_counts = Counter(tokens)
            total_tokens = len(tokens) if tokens else 1

            for word, count in token_counts.items():
                if word in self.vocab:
                    idx = self.vocab[word]
                    tf = count / total_tokens         # Term Frequency
                    idf = self.idf.get(word, 1)
                    matrix[i, idx] = tf * idf        # TF-IDF score

        return matrix

    def fit_transform(self, documents):
        """Fit then transform in one step."""
        return self.fit(documents).transform(documents)


# Build the embedding pipeline and embed all chunks
print('=== Building Embedding Pipeline ===')
embedder = TFIDFEmbedder(max_vocab=3000)
chunk_matrix = embedder.fit_transform(chunks)
print(f'\nEmbedding complete!')
print(f'Matrix shape: {chunk_matrix.shape}  ({chunk_matrix.shape[0]} chunks × {chunk_matrix.shape[1]} features)')

=== Building Embedding Pipeline ===
Building vocabulary and computing IDF scores...
Vocabulary size: 1815 words

Embedding complete!
Matrix shape: (137, 1815)  (137 chunks × 1815 features)


---
## Step 4: Custom Vector Store

In [20]:
# Name: Maureen Amago | Index: 10022200180

class CustomVectorStore:
    """
    Custom in-memory vector database.
    Stores chunk embeddings and performs top-k similarity search.
    """
    def __init__(self, chunks, embedder):
        self.chunks = chunks
        self.embedder = embedder
        self.vectors = embedder.transform(chunks)  # shape: (num_chunks, vocab_size)
        print(f'Vector Store ready: {len(chunks)} chunks stored.')

    def _cosine_similarity(self, vec_a, vec_b):
        """Compute cosine similarity between two vectors."""
        dot = np.dot(vec_a, vec_b)
        norm_a = np.linalg.norm(vec_a)
        norm_b = np.linalg.norm(vec_b)
        if norm_a == 0 or norm_b == 0:
            return 0.0
        return dot / (norm_a * norm_b)

    def search(self, query, k=2):
        """Search for the top-k most relevant chunks for a given query."""
        # Embed the query
        query_vec = self.embedder.transform([query])[0]

        # Compute similarity between query and every chunk
        scores = np.array([
            self._cosine_similarity(query_vec, self.vectors[i])
            for i in range(len(self.chunks))
        ])

        # Get top-k indices sorted by similarity (highest first)
        top_k_indices = np.argsort(scores)[-k:][::-1]

        results = []
        for idx in top_k_indices:
            results.append({
                'chunk_index': int(idx),
                'similarity_score': round(float(scores[idx]), 4),
                'text': self.chunks[idx]
            })
        return results


# Initialize the vector store
vector_store = CustomVectorStore(chunks, embedder)
print('Custom Vector Store initialized!')

Vector Store ready: 137 chunks stored.
Custom Vector Store initialized!


---
## Step 5: Top-K Retrieval & Similarity Scoring

In [21]:
# Name: Maureen Amago | Index: 10022200180

good_query = "tax revenue mobilisation 2025"

print(f'Query: "{good_query}"')
print('=' * 60)

results = vector_store.search(good_query, k=2)

for i, r in enumerate(results):
    print(f'\nResult #{i+1}')
    print(f'  Chunk Index     : {r["chunk_index"]}')
    print(f'  Similarity Score: {r["similarity_score"]}  (closer to 1.0 = more relevant)')
    print(f'  Text            : {r["text"]}')
    print('-' * 60)

Query: "tax revenue mobilisation 2025"

Result #1
  Chunk Index     : 18
  Similarity Score: 0.2481  (closer to 1.0 = more relevant)
  Text            : ls and (2025-2028) Medium Term Projections ....................................................................................... 185 Appendix 7B: Non-Tax Revenue / Internally Generated Funds (NTR/IGF) 2024 Proj Vs Actuals and 2025 Projections (GH '000) ....................................................................................................... 187 Appendix 8A: Non-Tax Revenue / Internally Generated Funds (NTR/IGF) 2025-2028 Medium-Term Estimates (GH '000) ...........................
------------------------------------------------------------

Result #2
  Chunk Index     : 36
  Similarity Score: 0.1656  (closer to 1.0 = more relevant)
  Text            : ................................................ 44 Figure 11: 2025 Resource Mobilisation (in GH  Million)............................................................. 58 

---
## Step 6: Failure Case — When Retrieval Gets It Wrong

**Problem:** Budget documents use formal language ("expenditure", "fiscal policy"). 
If a user asks using informal everyday words, the system fails because those words don't appear in the document — so the similarity score is very low and the returned chunk is not truly relevant.

In [22]:
# Name: Maureen Amago | Index: 10022200180

failing_query = "money the government is spending"

print(f'Failing Query: "{failing_query}"')
print('=' * 60)

failed_results = vector_store.search(failing_query, k=1)
for r in failed_results:
    print(f'Similarity Score : {r["similarity_score"]}  <- very low, means poor match')
    print(f'Retrieved Text   : {r["text"]}')

print('\n--- WHY IT FAILED ---')
print('Words like "money" and "spending" are everyday words.')
print('The budget PDF uses formal terms like "revenue", "expenditure", "fiscal".')
print('TF-IDF cannot match them because they have completely different vectors.')

Failing Query: "money the government is spending"
Similarity Score : 0.1304  <- very low, means poor match
Retrieved Text   : r. Speaker, we have inherited a precarious economic condition beset by daunting fiscal challenges characterized by large accumulation of MDA arrears/payables, energy sector financing shortfalls, and fiscal risks from the cocoa and financial sectors. Weak commitment control system and reckless public spending have reversed the progress made in fiscal consolidation under the IMF -supported Programme which commenced in 2023. Status of IMF-Supported Programme 64. Mr. Speaker, despite the gains made 

--- WHY IT FAILED ---
Words like "money" and "spending" are everyday words.
The budget PDF uses formal terms like "revenue", "expenditure", "fiscal".
TF-IDF cannot match them because they have completely different vectors.


---
## Step 7: Fix — Query Expansion

Before sending the query to the vector store, we detect informal words and **expand** the query with the correct formal synonyms. The enriched query then produces a much better embedding match.

In [23]:
# Name: Maureen Amago | Index: 10022200180

def expand_query(query):
    """
    Expands informal queries with formal budget-domain synonyms.
    """
    synonym_map = {
        'spending':    'expenditure allocation fiscal disbursement',
        'money':       'revenue funds budget finance',
        'getting cash':'revenue tax collection income mobilisation',
        'jobs':        'employment labour workforce',
        'farming':     'agriculture cocoa fisheries crops',
        'debt':        'borrowing loan liability obligations',
        'poor people': 'vulnerable households social protection poverty'
    }
    q = query.lower()
    extra = []
    for key, expansion in synonym_map.items():
        if key in q:
            extra.append(expansion)
    return query + ' ' + ' '.join(extra) if extra else query


# Apply the fix
expanded_query = expand_query(failing_query)

print(f'Original Query : {failing_query}')
print(f'Expanded Query : {expanded_query}')
print('\n=== RETRIEVAL WITH EXPANDED QUERY ===')

fixed_results = vector_store.search(expanded_query, k=1)
for r in fixed_results:
    print(f'Similarity Score : {r["similarity_score"]}  <- now much higher!')
    print(f'Retrieved Text   : {r["text"]}')

Original Query : money the government is spending
Expanded Query : money the government is spending expenditure allocation fiscal disbursement revenue funds budget finance

=== RETRIEVAL WITH EXPANDED QUERY ===
Similarity Score : 0.2823  <- now much higher!
Retrieved Text   : dix 3C: - Economic Classification of Central Gov't Expenditure ............................................ 136 Appendix 4A: MDA Expenditure Allocation (GH )   2025 ............................................................. 137 Appendix 4B: MDA Expenditure Allocation (GH )   2026 ............................................................. 155 Appendix 4C: MDA Expenditure Allocation (GH )   2027 ............................................................. 161 Appendix 4D: MDA Expenditure Al


---
## Step 8: Side-by-Side Comparison Summary

In [24]:
# Name: Maureen Amago | Index: 10022200180

score_before = vector_store.search(failing_query, k=1)[0]['similarity_score']
score_after  = vector_store.search(expand_query(failing_query), k=1)[0]['similarity_score']

summary = pd.DataFrame({
    'Method':           ['Without Query Expansion (Failure)', 'With Query Expansion (Fix)'],
    'Similarity Score': [score_before, score_after],
    'Result':           ['Poor — low score, irrelevant chunk', 'Good — higher score, relevant chunk']
})

display(summary)
print(f'\nImprovement: score went from {score_before} → {score_after}')
print('Query Expansion successfully fixed the retrieval failure!')

,Method,Similarity Score,Result
0,Without Query Expansion (Failure),0.1304,"Poor — low score, irrelevant chunk"
1,With Query Expansion (Fix),0.2823,"Good — higher score, relevant chunk"



Improvement: score went from 0.1304 → 0.2823
Query Expansion successfully fixed the retrieval failure!
